# Lab 2 — Write, update, forget, reflect (EP12)

Lab 1 ended with a problem: the same memory in different words slipped past the duplicate check and crowded out the report topic.
In this lab **the model helps manage memory**, all four operations (§17.4)

**What you'll do**
1. **Write:** the model rates importance 1–10 → store only what clears the threshold τ
2. **Extract short facts** (atomic facts), the Mem0 way (§17.11.2)
3. **Update:** ADD / DUPLICATE / UPDATE — an old fact isn't deleted, it's closed with a `valid_until` time
4. **Forget:** delete when the user asks (§17.5 privacy)
5. **Reflect:** three real failures → one lesson → the next task passes

Every answer from the model comes from a real run.

> Source: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §17.4, §17.5, §17.11.2 — CC BY-SA 4.0

> ✏️ **Starter notebook — you write the key parts**
>
> Most of the code is ready. Write only the **5 places marked `✏️ TODO`** (replace `___` or `...` with your code) — each one gives the time in the video where that code is explained (▶ m:ss). Watch the video first, then try it yourself without opening the solution.
>
> Run every cell from top to bottom. A `NameError` or `TypeError` in a cell you have not finished yet is normal.
>
> Stuck? The solution is **`ep12-lab2-memory-operations.en.ipynb`** (the same code as in the video).

## 0. Setup

| You need | Details |
|---|---|
| Python 3.10 or later | or Google Colab, which already has it |
| An OpenRouter API key | sign up at openrouter.ai → Keys → Create Key, then add credit (this lab costs less than 0.1 US cents) |
| Packages | `openai`, `python-dotenv`, `numpy` (installed in the next cell) |

**Adding the API key**
- **Colab:** the 🔑 Secrets icon → name `OPENROUTER_API_KEY` → turn on *Notebook access*
- **Your own machine:** create a `.env` file next to the notebook with the line `OPENROUTER_API_KEY=sk-or-...`

⚠️ Never put the key in the code, and never share your `.env` file

In [ ]:
%pip install -q "openai>=1.50" python-dotenv numpy

## 1. Models and helpers

Gemma 4 26B (Makora) answers · Qwen3 Embedding 8B (Nebius) embeds — as in Lab 1 (why these models: see Lab 1)
New: `ask_json` — ask for JSON that follows a schema (structured output, from EP02)

In [ ]:
import os, json, re
from dataclasses import dataclass, field
from datetime import datetime, timedelta
import numpy as np
from openai import OpenAI

def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

MODEL = "google/gemma-4-26b-a4b-it"
EXTRA = {"reasoning": {"enabled": False},
         "provider": {"order": ["makora"], "allow_fallbacks": False}}
EMBED_MODEL = "qwen/qwen3-embedding-8b"
EMBED_EXTRA = {"provider": {"order": ["nebius"], "allow_fallbacks": False}}

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key(), max_retries=5)
COST = {"chat": 0.0, "embed": 0.0}

def chat(messages):
    r = client.chat.completions.create(model=MODEL, temperature=0, extra_body=EXTRA,
                                       messages=messages)
    COST["chat"] += r.usage.cost
    return r.choices[0].message.content

def embed(text):
    r = client.embeddings.create(model=EMBED_MODEL, input=[text], extra_body=EMBED_EXTRA)
    COST["embed"] += r.usage.cost
    v = np.array(r.data[0].embedding)
    return v / np.linalg.norm(v)

In [ ]:
def ask_json(prompt, schema):
    r = client.chat.completions.create(
        model=MODEL, temperature=0, extra_body=EXTRA,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_schema",
                         "json_schema": {"name": "reply", "strict": True, "schema": schema}})
    COST["chat"] += r.usage.cost
    return json.loads(r.choices[0].message.content)

## 2. The memory store + `valid_until`

Lab 1's `MemoryEntry` with one more field: **`valid_until`** — a replaced fact isn't deleted, it's closed with a time
("keep both with timestamps" — one of the book's three options when memories conflict, §17.4.1)
A simpler search: similarity only, over the memories that are still true

In [ ]:
@dataclass
class MemoryEntry:
    content: str
    embedding: np.ndarray
    timestamp: datetime = field(default_factory=datetime.now)
    importance: float = 0.5
    valid_until: datetime | None = None          # set when a newer fact replaces this one

# ✏️ TODO 1 (▶ video 1:35): return only the memories that are still true (valid_until is None)

class MemoryStore:
    def __init__(self):
        self.entries = []

    def current(self):
        return ___

    def search(self, q, k=3, everything=False):
        pool = self.entries if everything else self.current()
        if not pool:
            return []
        sims = np.stack([e.embedding for e in pool]) @ q
        return [(sims[i], pool[i]) for i in np.argsort(-sims)[:k]]

mem = MemoryStore()

## 3. Write: the model rates importance

`Write(e) = 1[importance(e) > τ]` (§17.4.1) — using "LLM self-assessment" on a 1–10 scale · store if ≥ τ = 5

In [ ]:
IMPORTANCE = {"type": "object", "additionalProperties": False, "required": ["reason", "importance"],
              "properties": {"reason": {"type": "string"},
                             "importance": {"type": "integer", "minimum": 1, "maximum": 10}}}

def rate(message):
    return ask_json("You manage the long-term memory of a research assistant. "
                    "Facts about the user's work, deadlines and preferences matter in future sessions; "
                    "small talk and personal details unrelated to the work do not. "
                    "Rate from 1 to 10 how important it is to remember this user message "
                    "(10 = essential, 1 = small talk).\nMessage: " + message,
                    IMPORTANCE)["importance"]

## 4. Extract facts, then decide ADD / DUPLICATE / UPDATE

The Mem0 way (§17.11.2): extract facts from the conversation automatically, then compare with the 3 closest memories
| action | what it does |
|---|---|
| ADD | new information → store |
| DUPLICATE | already known → don't store (fixes Lab 1's problem) |
| UPDATE | replaces an old fact that is no longer true → close the old one with `valid_until`, store the new one |

In [ ]:
FACTS = {"type": "object", "additionalProperties": False, "required": ["facts"],
         "properties": {"facts": {"type": "array", "items": {"type": "string"}}}}

def extract(message):
    return ask_json("Extract the facts about the user worth remembering from this message, "
                    "as short standalone sentences that start with 'The user'. "
                    "One fact per sentence.\nMessage: " + message, FACTS)["facts"]

DECISION = {"type": "object", "additionalProperties": False, "required": ["action", "target"],
            "properties": {"action": {"type": "string", "enum": ["ADD", "DUPLICATE", "UPDATE"]},
                           "target": {"type": "integer"}}}

def decide(fact, candidates):
    listing = "\n".join(f"{i}. {e.content}" for i, (_, e) in enumerate(candidates)) or "(none)"
    return ask_json(f"New fact about the user: {fact}\nExisting memories:\n{listing}\n"
                    "Choose one action:\n"
                    "ADD - the new fact is new information.\n"
                    "DUPLICATE - an existing memory already says the same thing.\n"
                    "UPDATE - an existing memory gives a different value for the same detail "
                    "(for example an old deadline and a new deadline), so it is no longer true.\n"
                    "Set target to the number of that existing memory, or -1 for ADD.", DECISION)

In [ ]:
# ✏️ TODO 2 (▶ video 3:36): if the score is under the threshold tau, don't remember it
def remember(message, when=None, tau=5):
    score = rate(message)
    print(f"[{score:>2}] {message}")
    if ___:
        return                                         # not worth remembering
    for fact in extract(message):
        emb = embed(fact)
        candidates = mem.search(emb)
        d = decide(fact, candidates)
        old = candidates[d["target"]][1] if 0 <= d["target"] < len(candidates) else None
        print(f"     {d['action']:<9} {fact}" + (f"  ← {old.content}" if old else ""))
        # ✏️ TODO 3 (▶ video 3:56): for UPDATE, close the old memory with the time when, or now
        if d["action"] == "DUPLICATE":
            continue
        if d["action"] == "UPDATE" and old:
            ___
        mem.entries.append(MemoryEntry(fact, emb, when or datetime.now(), score / 10))

## 5. Monday → Tuesday

Six messages from Monday (simulated time: yesterday), then Tuesday's new message

In [ ]:
yesterday = datetime.now() - timedelta(days=1)
monday = ["Hi! I'm writing a report on transformers for time series.",
          "Please keep your answers short.",
          "Thanks!",
          "By the way, I live in Bangkok.",
          "My report deadline is Friday.",
          "Short answers, please. Bullet points are fine."]
for message in monday:
    remember(message, when=yesterday)

In [ ]:
remember("My report deadline moved to next Wednesday.")
print()
for e in mem.entries:
    until = e.valid_until.strftime("%a %d %b") if e.valid_until else "now"
    print(f"{e.timestamp:%a %d %b} → {until:<10}  {e.importance:.1f}  {e.content}")

## 6. Answering with memory

In [ ]:
def answer(question, k=3):
    hits = mem.search(embed(question), k)
    for sim, e in hits:
        print(f"{sim:.2f}  {e.content}")
    system = "What you remember about this user (follow their preferences):\n" + \
             "\n".join(f"- {e.content}" for _, e in hits)
    return chat([{"role": "system", "content": system}, {"role": "user", "content": question}])

print(answer("When is my report due?"))

In [ ]:
print(answer("Any new papers for my report?")[:500])

## 7. Forget: delete when the user asks

The book (§17.5 Privacy): users must be able to **inspect and delete** their memories — both the current ones **and the history**

In [ ]:
FORGET = {"type": "object", "additionalProperties": False, "required": ["delete"],
          "properties": {"delete": {"type": "array", "items": {"type": "integer"}}}}

def forget(request):
    candidates = mem.search(embed(request), k=5, everything=True)
    listing = "\n".join(f"{i}. {e.content}" for i, (_, e) in enumerate(candidates))
    out = ask_json(f"The user asks: {request}\nMemories:\n{listing}\n"
                   "Which memories must be deleted to respect this request? Give their numbers.", FORGET)
    # ✏️ TODO 4 (▶ video 6:12): remove the memories the model chose from mem.entries
    for i in out["delete"]:
        print("deleted:", candidates[i][1].content)
        ___

forget("Please forget everything about my deadline.")

## 8. Reflect: from failures to a lesson (§17.4.4)

The model writes three functions, and we test each on an empty list → the results become **episodic memory**
→ reflect them into one lesson (**semantic memory**) → a new task, without / with the lesson

⚠️ This part runs code the model wrote (`exec`) on our own machine — fine for this experiment only; real systems run it in a sandbox

In [ ]:
def try_empty(code, fn):
    ns = {}
    try:
        exec(code, ns)
        return f"ok → {ns[fn]([])!r}"
    except Exception as e:
        return f"{type(e).__name__}: {e}"

def attempt(fn, desc, lessons=()):
    messages = [{"role": "user", "content": f"Write a Python function {fn}(xs) that {desc}. Code only."}]
    if lessons:
        messages.insert(0, {"role": "system", "content": "Lessons from your past attempts:\n" +
                            "\n".join("- " + l for l in lessons)})
    code = re.search(r"```(?:python)?\n(.*?)```", chat(messages), re.S).group(1)
    return try_empty(code, fn)

In [ ]:
episodes = []
for fn, desc in [("mean", "returns the mean of a list of numbers"),
                 ("first_last", "returns a tuple of the first and last items of a list"),
                 ("range_of", "returns max minus min of a list of numbers")]:
    episodes.append(f"Task: write {fn}(xs). Test {fn}([]): {attempt(fn, desc)}")
    print(episodes[-1])

In [ ]:
INSIGHT = {"type": "object", "additionalProperties": False, "required": ["insight"],
           "properties": {"insight": {"type": "string"}}}

def reflect(episodes):
    return ask_json("These are past attempts of a coding agent:\n" +
                    "\n".join("- " + e for e in episodes) +
                    "\nWhat one general lesson should the agent remember for future tasks? "
                    "One sentence.", INSIGHT)["insight"]

# ✏️ TODO 5 (▶ video 7:36): store the lesson in mem as a MemoryEntry (embed it too), importance 0.9

insight = "[INSIGHT] " + reflect(episodes)
___
print(insight)

In [ ]:
task = ("largest", "returns the largest number in a list")
hits = mem.search(embed(f"Write a Python function {task[0]}(xs) that {task[1]}."), k=1)
print(f"retrieved: {hits[0][1].content}\n")
for run in range(3):
    print("no memory   ", attempt(*task))
    print("with lesson ", attempt(*task, lessons=[hits[0][1].content]))

In [ ]:
print({k: f"${v:.5f}" for k, v in COST.items()})

## Summary

| Operation | How | What we saw |
|---|---|---|
| Write | the model rates 1–10, τ = 5 | "Thanks!" 1 · "I live in Bangkok" 1 → not remembered · the deadline 10 |
| Extract | atomic facts, the Mem0 way | one message → two facts |
| Update | ADD / DUPLICATE / UPDATE | the second request for short answers → DUPLICATE · Wednesday → UPDATE Friday (kept as history) |
| Read | search only memories still true | "When is my report due?" → next Wednesday · the report topic is back in the top three |
| Forget | delete on request, history included | deletes both Wednesday and Friday |
| Reflect | episodic → insight → semantic | `largest([])` without the lesson: fails 3/3 · with it: passes 3/3 |

**Exercises**
1. Set `tau=8` — which messages disappear, and can Tuesday still answer?
2. Remove the criteria sentence ("Facts about the user's work…") from `rate` and run it several times — is the deadline's score steady?
3. Write `history(question)`, which searches with `everything=True`, and answer "When was my report due before?" (do it before `forget`)

**Next — Lab 3:** an agent with three memory tiers (hot / warm / cold) that calls its memory through tools — and we measure it